In [ ]:
import os
import matplotlib.pyplot as plt 
import numpy as np 
from os.path import join as opj
from IPython.display import Video
import torch
import tqdm
from diffusers import CogVideoXPipeline
from IPython.display import display, Image as IPImage

In [2]:
train_models = False

In [ ]:
base_path = "/your_path"
sub = f"subj0{1,2,3}"

In [ ]:
fmri_train = torch.load(opj(base_path, f"{sub}_train_fmri.pt")) 
fmri_test = torch.load(opj(base_path, f"{sub}_test_fmri.pt"))

video_train = torch.load(opj(base_path, f"GT_train_3fps.pt"))
video_test = torch.load(opj(base_path, f"GT_test_3fps.pt"))

In [ ]:
video_train.shape

In [ ]:
caption_train = torch.load(opj(base_path, f"GT_train_caption.pt"))
caption_test = torch.load(opj(base_path, f"GT_test_caption.pt"))

In [ ]:
print(type(caption_train))

In [ ]:
fmri_train.shape, fmri_test.shape, video_train.shape, video_test.shape

In [ ]:
fmri_train_avg = fmri_train.mean(dim=1)
fmri_test_avg = fmri_test.mean(dim=1)

fmri_train_avg.mean(), fmri_train_avg.std(), fmri_test_avg.mean(), fmri_test_avg.std() 

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=(10, 10))
idx = 0
for i, ax in enumerate(ax.flatten()):
    ax.imshow(video_train[idx][i].permute(1,2,0).numpy())
    ax.set_title(f"frame {i}")
    ax.axis("off")
print(caption_train[idx])

In [ ]:
video_pipe = CogVideoXPipeline.from_pretrained(
    "THUDM/CogVideoX-5b",
    torch_dtype=torch.bfloat16
).to("cuda:2")

In [16]:

video_pipe.vae.enable_slicing()
video_pipe.vae.enable_tiling()

## Encode prompt

In [ ]:
batch = 32

train_prompt_embeds = []
train_pooled_prompt_embeds = []
train_video_prompt_embeds = []

with torch.no_grad():
    for i in tqdm.trange(0, len(caption_train), batch):
       
        video_prompt_embeds, negative_video_prompt_embeds, = video_pipe.encode_prompt(caption_train[i:i+batch].tolist())
        
        train_video_prompt_embeds.append(video_prompt_embeds.cpu())

In [ ]:
batch = 32

test_prompt_embeds = []
test_pooled_prompt_embeds = []
test_video_prompt_embeds = []

with torch.no_grad():
    for i in tqdm.trange(0, len(caption_test), batch):
        
        video_prompt_embeds, negative_video_prompt_embeds, = video_pipe.encode_prompt(caption_test[i:i+batch].tolist())

        test_video_prompt_embeds.append(video_prompt_embeds.cpu())

In [ ]:


test_video_prompt_embeds = torch.cat(test_video_prompt_embeds)
train_video_prompt_embeds = torch.cat(train_video_prompt_embeds)

print(train_video_prompt_embeds.shape, test_video_prompt_embeds.shape)

In [ ]:
from himalaya.ridge import Ridge, RidgeCV
from himalaya.backend import set_backend
import torch
set_backend("torch_cuda")
brain_to_video_prompts = []
train_models=False
new_path='/new_path'

if train_models:
    
    for i in tqdm.trange(train_video_prompt_embeds.shape[1]):
        brain_to_video_prompts.append(RidgeCV(alphas=[1,10,1e2,1e3,1e4]).fit(fmri_train_avg, train_video_prompt_embeds[:, i].float()))
else:
    for i in tqdm.trange(train_video_prompt_embeds.shape[1]):
        model = torch.load(opj(new_path, f"{sub}_video_model_{i}.pt"))
        brain_to_video_prompts.append(model)
    


In [ ]:

if train_models:
    
    for i, model in enumerate(brain_to_video_prompts):
        torch.save(model, opj(new_path, f"{sub}_video_model_{i}.pt"))
    print("Model saved",sub)
else:
    print("model loaded")

In [ ]:

predicted_video_embeds = torch.stack([model.predict(fmri_test_avg) for model in tqdm.tqdm(brain_to_video_prompts)], dim=1)

In [ ]:

predicted_train_video_embeds = torch.stack([model.predict(fmri_train_avg) for model in tqdm.tqdm(brain_to_video_prompts)], dim=1)



In [ ]:

video_prompt_embeds_predicted_mean = predicted_train_video_embeds.mean(dim=0).cpu()
video_prompt_embeds_predicted_std = predicted_train_video_embeds.std(dim=0).cpu()

expected_video_prompt_embeds_mean  = train_video_prompt_embeds.mean(dim=0).cpu()
expected_video_prompt_embeds_std = train_video_prompt_embeds.std(dim=0).cpu()


In [ ]:

def renormalize_video_prompt_embeds(predicted):

    predicted_std = (predicted - video_prompt_embeds_predicted_mean)/video_prompt_embeds_predicted_std
    return predicted_std * expected_video_prompt_embeds_std + expected_video_prompt_embeds_mean



In [ ]:
adjusted_predicted_video_prompt_embeds = renormalize_video_prompt_embeds(predicted_video_embeds)

In [ ]:
adjusted_predicted_video_prompt_embeds.mean(), adjusted_predicted_video_prompt_embeds.std(), test_prompt_embeds.mean(), test_prompt_embeds.std()

In [ ]:

idx = 20
n_generations = 5

videos_generated = []

base_seed = 42

for i in range(n_generations):

    generator = torch.Generator(device="cuda").manual_seed(base_seed + i)

    video = video_pipe(
        prompt_embeds=adjusted_predicted_video_prompt_embeds[idx]
            .unsqueeze(0)
            .bfloat16()
            .to("cuda"),
        num_videos_per_prompt=1,
        num_inference_steps=45,
        num_frames=49,
        guidance_scale=6,
        generator=generator,
    ).frames


    videos_generated.append(video[0])